# fy-m1 00 — base-model baseline (T4)
One candidate per run. **Runtime → Run all**; the notebook picks the first candidate in `CANDIDATES` whose
results file on `Fan1000-Prog/fyM1-ckpt` (`results/baseline/<slug>.json`) still misses a required section, and
skips sections already saved there, so a rerun after a disconnect resumes. Restart the runtime between candidates
(**Runtime → Restart session**, then **Run all**). Set `FORCE` to run one candidate explicitly.

Measures, all 4-bit: Belebele `plt_Latn` (900, 0-shot, next-token over 1–4), FLORES+ `plt_Latn`↔`eng_Latn`
devtest first 200 chrF++ (greedy), tokens per Malagasy word (1,000 Wikipedia sentences), 20 greeting answers,
and a short QLoRA smoke train (seq 2048, bs 1 × grad-accum 8) for peak VRAM and sec/step.
Runs on Colab or Kaggle (T4). Needs the `HF_TOKEN` secret (Colab: Secrets pane with notebook access; Kaggle: Add-ons → Secrets) and code synced to `fyM1-ckpt` `code/` (`scripts/sync_code_to_hub.py`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U unsloth unsloth_zoo sacrebleu

In [ ]:
CANDIDATES = {
    # slug: (hub id, QLoRA smoke steps, push 4-bit weights to fyM1-ckpt base-4bit/<slug>/, base model?)
    # Order = run order. 4-bit push off: on T4 Unsloth keeps gemma4 in fp32 and save_pretrained stalled at
    # ~11.5/12.7 GB CPU RAM. Belebele: plain-text completion for all; chat-template score only for instruct models.
    # Gemma 3 is quantized on load like Gemma 4: Unsloth's pre-quantized bnb-4bit checkpoint keeps many layers
    # unquantized and used 11.9 GB of the T4 right after loading (OOM at the first Belebele batch).
    "gemma-3-12b-it": ("google/gemma-3-12b-it", 15, False, False),
    "afriqueqwen3.5-9b-instruct": ("McGill-NLP/AfriqueQwen3.5-9B-50Langs-Instruct-v1", 15, False, False),
    "gemma-4-12b-it": ("google/gemma-4-12B-it", 15, False, False),
    "afriqueqwen-8b": ("McGill-NLP/AfriqueQwen-8B", 15, False, True),
}
FORCE = None  # e.g. "afriqueqwen-8b"
REPO = "Fan1000-Prog/fyM1-ckpt"
BELEBELE_BATCH, GEN_BATCH, FLORES_N, TPW_SENTENCES = 4, 8, 200, 1000
FALLBACK_GREETINGS = ["Sali ahn", "De ahoana?", "Manao ahoana ianao?", "De aona ry zalahy?", "Salut, mbola tsy natory ve?"]

In [ ]:
# Runtime bootstrap (Colab or Kaggle). The HF token is needed before fym1 can be downloaded, so this cell
# carries a verbatim copy of fym1.runtime.hf_token and its helpers (checked by tests/test_notebook_runtime.py).
import os
import tempfile

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle's T4 runtime may preset "0,1"; always use one GPU
CODE_DIR = tempfile.mkdtemp(prefix="fym1code-")  # fresh each run: nothing stale on sys.path

MISSING_TOKEN = ("no HF_TOKEN found: on Kaggle attach it under Add-ons → Secrets; on Colab add it in the "
                 "Secrets pane with notebook access; elsewhere set the HF_TOKEN environment variable")


def on_kaggle() -> bool:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return True
    try:
        import kaggle_secrets  # noqa: F401  (only installed in Kaggle kernels)
    except ImportError:
        return False
    return True


def _secret(get) -> str | None:
    try:
        return get("HF_TOKEN")
    except Exception:  # missing or not attached: the caller raises one clear error; never echo details
        return None


def hf_token() -> str:
    """HF token from Kaggle secrets, else Colab secrets, else the HF_TOKEN env var. Never print it."""
    if on_kaggle():
        from kaggle_secrets import UserSecretsClient

        token = _secret(UserSecretsClient().get_secret)
    else:
        try:
            from google.colab import userdata
        except ImportError:
            token = os.environ.get("HF_TOKEN")
        else:
            token = _secret(userdata.get)
    if not token:
        raise RuntimeError(MISSING_TOKEN)
    return token

In [ ]:
import sys, json, time, datetime
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from huggingface_hub.errors import EntryNotFoundError

os.environ["HF_TOKEN"] = hf_token()  # never print it
api = HfApi()
code_dir = snapshot_download(REPO, allow_patterns=["code/**"], local_dir=CODE_DIR)
sys.path.insert(0, f"{code_dir}/code/src")
sha_file = f"{code_dir}/code/SHA"
CODE_SHA = open(sha_file).read().strip() if os.path.exists(sha_file) else "unknown"  # git commit of the synced code
from fym1.eval import baseline as bl
from fym1.eval.batching import StillOutOfMemory, run_batched
from fym1.train.chat_format import CHAT_TEMPLATE_KWARGS
from fym1.runtime import output_dir, scratch_dir

OUTPUT, SCRATCH = output_dir(), scratch_dir()  # small wanted results vs big throwaway files


def load_results(slug):
    try:
        return json.load(open(hf_hub_download(REPO, f"results/baseline/{slug}.json", force_download=True)))
    except EntryNotFoundError:
        return {}


def save_results(slug, results):
    for name, section_result in results.items():  # each section records the code and FORCE of its run
        if name not in ("model", "env") and isinstance(section_result, dict):
            section_result.setdefault("code_sha", CODE_SHA)
            section_result.setdefault("force", FORCE)
    path = f"{OUTPUT}/{slug}.json"
    with open(path, "w") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    api.upload_file(path_or_fileobj=path, path_in_repo=f"results/baseline/{slug}.json", repo_id=REPO,
                    commit_message=f"baseline {slug}: {', '.join(k for k in results if k not in ('model', 'env'))}")


SLUG = FORCE or next((s for s in CANDIDATES if bl.missing_sections(load_results(s), CANDIDATES[s][3])), None)
assert SLUG, "all candidates complete"
HUB_ID, SMOKE_STEPS, PUSH_4BIT, BASE = CANDIDATES[SLUG]
R = load_results(SLUG)
R.pop("done", None)  # completeness is missing_sections(), not a stored flag
for name, section_result in R.items():  # sections saved before code SHA / FORCE were recorded
    if name not in ("model", "env") and isinstance(section_result, dict):
        section_result.setdefault("code_sha", "unrecorded")
        section_result.setdefault("force", "unrecorded")
R.setdefault("model", {"slug": SLUG, "hub_id": HUB_ID, "quant": "bnb 4-bit nf4 (Unsloth load_in_4bit)"})
print("CANDIDATE", SLUG, "| missing:", bl.missing_sections(R, BASE))

In [ ]:
import torch
from unsloth import FastModel

RETRIED = {"items": 0}  # items re-run one at a time after an OOM batch, in the current section


def section(name):
    print(f"== {name}: {'skip (saved)' if name in R else 'run'}")
    RETRIED["items"] = 0
    return name not in R

t0 = time.time()
try:
    # use_exact_model_name: else Unsloth's name map swaps google/gemma-3-12b-it for its own bnb-4bit checkpoint
    model, processor = FastModel.from_pretrained(HUB_ID, max_seq_length=2048, load_in_4bit=True, full_finetuning=False,
                                                 use_exact_model_name=True)
except Exception as e:  # gated/403 or unsupported architecture: record and move on
    R["skipped"] = {"error": f"{type(e).__name__}: {str(e)[:300]}"}  # missing_sections() then treats it as complete
    save_results(SLUG, R)
    raise
tok = getattr(processor, "tokenizer", processor)
tok.padding_side = "left"
R["env"] = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "model_dtype": str(model.dtype),
            "loaded_from": model.config._name_or_path, "code_sha": CODE_SHA,
            "vram_after_load_gb": round(torch.cuda.memory_allocated() / 2**30, 2),
            "load_s": round(time.time() - t0), "utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
print(R["env"], "vram after load GB:", round(torch.cuda.memory_allocated() / 2**30, 2))


def chat(prompt):
    return tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                   add_generation_prompt=True, **CHAT_TEMPLATE_KWARGS)


def encode(texts, special_tokens=False):
    return tok(texts, return_tensors="pt", padding=True, add_special_tokens=special_tokens).to("cuda")


def batched(items, batch_size, run):
    """On CUDA OOM: empty the cache and retry that batch one item at a time; StillOutOfMemory if that fails."""
    def count(n):
        RETRIED["items"] += n

    return run_batched(items, batch_size, run, is_oom=lambda e: isinstance(e, torch.OutOfMemoryError),
                       clear=torch.cuda.empty_cache, on_retry=count)


@torch.inference_mode()
def generate(prompts, max_new_tokens, sample):
    kw = dict(do_sample=True, temperature=0.7, top_p=0.9) if sample else dict(do_sample=False)

    def run(batch):
        enc = encode([chat(p) for p in batch])
        ids = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **kw)
        return tok.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

    return batched(prompts, GEN_BATCH, run)


@torch.inference_mode()
def last_logits(enc):
    try:
        return model(**enc, logits_to_keep=1).logits[:, -1]
    except TypeError:  # forward without logits_to_keep
        return model(**enc).logits[:, -1]

In [ ]:
if PUSH_4BIT and section("base_4bit"):
    t0 = time.time()
    d = f"{SCRATCH}/base-4bit/{SLUG}"
    try:
        model.save_pretrained(d)
        processor.save_pretrained(d)
        api.upload_folder(folder_path=d, path_in_repo=f"base-4bit/{SLUG}", repo_id=REPO,
                          commit_message=f"bnb-4bit weights of {HUB_ID}")
        R["base_4bit"] = {"path": f"base-4bit/{SLUG}", "upload_s": round(time.time() - t0)}
    except Exception as e:  # not fatal: evals still run
        R["base_4bit"] = {"error": f"{type(e).__name__}: {str(e)[:300]}"}
    save_results(SLUG, R)
    print(R["base_4bit"])

In [ ]:
from datasets import load_dataset

FastModel.for_inference(model)
if not BASE and section("belebele"):
    t0 = time.time()
    rows = load_dataset("facebook/belebele", "plt_Latn", split="test")
    digit_ids = [tok.encode(d, add_special_tokens=False)[-1] for d in "1234"]
    protocol = "0-shot, chat template, English instruction, argmax of next-token logits over '1'-'4'"
    try:
        scores = torch.tensor(batched([chat(bl.belebele_prompt(r)) for r in rows], BELEBELE_BATCH,
                                      lambda b: last_logits(encode(b))[:, digit_ids].float().tolist()))
        R["belebele"] = {"acc": round(bl.accuracy((scores.argmax(-1) + 1).tolist(), rows["correct_answer_num"]), 4),
                         "n": len(scores), "nan_rows": int(torch.isnan(scores).any(-1).sum()),
                         "seconds": round(time.time() - t0), "protocol": protocol, "retried_items": RETRIED["items"]}
    except StillOutOfMemory:
        R["belebele"] = {"oom": True, "protocol": protocol, "retried_items": RETRIED["items"]}
    save_results(SLUG, R)
print("BELEBELE", R.get("belebele"))

In [ ]:
if section("belebele_completion"):  # the one protocol for every model: plain text, no chat template
    t0 = time.time()
    rows = load_dataset("facebook/belebele", "plt_Latn", split="test")
    letter_ids = [tok.encode(" " + letter, add_special_tokens=False)[-1] for letter in "ABCD"]
    protocol = "0-shot plain-text completion (P/Q/A-D/Answer:), no chat template, argmax of next-token logits over ' A'-' D'"
    try:
        scores = torch.tensor(batched([bl.belebele_completion_prompt(r) for r in rows], BELEBELE_BATCH,
                                      lambda b: last_logits(encode(b, special_tokens=True))[:, letter_ids].float().tolist()))
        R["belebele_completion"] = {"acc": round(bl.accuracy((scores.argmax(-1) + 1).tolist(), rows["correct_answer_num"]), 4),
                                    "n": len(scores), "nan_rows": int(torch.isnan(scores).any(-1).sum()),
                                    "seconds": round(time.time() - t0), "protocol": protocol,
                                    "retried_items": RETRIED["items"]}
    except StillOutOfMemory:
        R["belebele_completion"] = {"oom": True, "protocol": protocol, "retried_items": RETRIED["items"]}
    save_results(SLUG, R)
print("BELEBELE_COMPLETION", R.get("belebele_completion"))

In [ ]:
if section("flores"):
    t0 = time.time()
    protocol = "devtest first 200, 0-shot chat prompt, greedy, max 256 new tokens, first line kept"
    try:
        mg = load_dataset("openlanguagedata/flores_plus", "plt_Latn", split="devtest")
        en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
        print("flores columns:", mg.column_names)
        mg_txt, en_txt = mg["text"][:FLORES_N], en["text"][:FLORES_N]
        if "id" in mg.column_names:
            assert mg["id"][:FLORES_N] == en["id"][:FLORES_N], "FLORES rows not aligned"
        hyp_mg = [bl.first_line(t) for t in generate([bl.translation_prompt(s, "English", "Malagasy") for s in en_txt], 256, False)]
        hyp_en = [bl.first_line(t) for t in generate([bl.translation_prompt(s, "Malagasy", "English") for s in mg_txt], 256, False)]
        R["flores"] = {"eng-plt_chrf++": round(bl.chrf_pp(hyp_mg, mg_txt), 2),
                       "plt-eng_chrf++": round(bl.chrf_pp(hyp_en, en_txt), 2), "n": FLORES_N,
                       "seconds": round(time.time() - t0), "protocol": protocol, "retried_items": RETRIED["items"]}
        save_results(SLUG, R)
    except StillOutOfMemory:
        R["flores"] = {"oom": True, "protocol": protocol, "retried_items": RETRIED["items"]}
        save_results(SLUG, R)
    except Exception as e:  # gated dataset not yet accepted: leave the section unsaved so a rerun retries it
        print("FLORES PENDING:", type(e).__name__, str(e)[:300])
print("FLORES", R.get("flores", "pending"))

In [ ]:
if section("tokens_per_word"):
    wiki = load_dataset("wikimedia/wikipedia", "20231101.mg", split="train").shuffle(seed=42)
    sents = bl.sample_sentences((r["text"] for r in wiki), n=TPW_SENTENCES)
    R["tokens_per_word"] = {"value": round(bl.tokens_per_word(sents, lambda s: len(tok.encode(s, add_special_tokens=False))), 3),
                            "n_sentences": len(sents), "source": "wikimedia/wikipedia 20231101.mg, shuffle seed 42, 5-60 words"}
    save_results(SLUG, R)
print("TPW", R["tokens_per_word"])

In [ ]:
if section("greetings"):
    try:
        rows = [json.loads(l) for l in open(hf_hub_download(REPO, "code/eval/prompts.jsonl"))]
        greetings = [r["prompt"] for r in rows if r["category"].startswith("greet")][:20]
        source = "eval/prompts.jsonl greetings, first 20"
    except EntryNotFoundError:
        greetings, source = FALLBACK_GREETINGS, "fallback made-up greetings"
    torch.manual_seed(42)
    sampling = "temp 0.7, top_p 0.9, max 128 new tokens, seed 42, no system prompt"
    try:
        answers = generate(greetings, 128, True)
        R["greetings"] = {"source": source, "sampling": sampling, "retried_items": RETRIED["items"],
                          "answers": [{"prompt": p, "answer": a.strip()} for p, a in zip(greetings, answers)]}
    except StillOutOfMemory:
        R["greetings"] = {"oom": True, "source": source, "sampling": sampling, "retried_items": RETRIED["items"]}
    save_results(SLUG, R)
for a in R["greetings"].get("answers", []):
    print(repr(a["prompt"]), "->", repr(a["answer"][:200]))

In [ ]:
if section("qlora_smoke"):
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback
    from datasets import Dataset

    wiki = load_dataset("wikimedia/wikipedia", "20231101.mg", split="train").shuffle(seed=7)
    chunks, buf = [], ""
    for r in wiki:  # ~2048-token chunks of Malagasy text: worst-case length for the fit test
        buf += r["text"] + "\n\n"
        if len(tok.encode(buf, add_special_tokens=False)) >= 2048:
            chunks.append(buf)
            buf = ""
            if len(chunks) == SMOKE_STEPS * 8:
                break
    multimodal = hasattr(model.config, "vision_config")
    lora = dict(r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=42, use_gradient_checkpointing="unsloth")
    if multimodal:
        model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
                                         finetune_attention_modules=True, finetune_mlp_modules=True, **lora)
    else:
        model = FastModel.get_peft_model(model, target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                                                "gate_proj", "up_proj", "down_proj"], **lora)
    step_times = []

    class StepTimer(TrainerCallback):
        def on_step_end(self, args, state, control, **kw):
            step_times.append(time.time())

    tok.padding_side = "right"
    compute_dtype = next(p.dtype for p in model.parameters() if p.is_floating_point())  # Unsloth may force fp32 on T4
    trainer = SFTTrainer(model=model, processing_class=tok, train_dataset=Dataset.from_dict({"text": chunks}),
                         callbacks=[StepTimer()],
                         args=SFTConfig(output_dir=f"{SCRATCH}/smoke_out", per_device_train_batch_size=1,
                                        gradient_accumulation_steps=8, max_steps=SMOKE_STEPS, learning_rate=2e-4,
                                        lr_scheduler_type="cosine", warmup_steps=3, logging_steps=1, optim="adamw_8bit",
                                        fp16=compute_dtype == torch.float16, bf16=False, max_length=2048,
                                        dataset_text_field="text",
                                        packing=False, save_strategy="no", report_to="none", seed=42))
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        try:
            trainer.train()
            stopped_early = None
        except KeyboardInterrupt:  # operator stop: what ran so far is still a valid measurement
            stopped_early = "interrupted by the operator"
        losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
        gaps = [b - a for a, b in zip(step_times, step_times[1:])]
        R["qlora_smoke"] = {"ok": True, "steps": len(step_times), "steps_planned": SMOKE_STEPS,
                            "stopped_early": stopped_early, "seq_len": 2048, "batch": "1 x grad-accum 8",
                            "tokens_per_step": 8 * 2048, "lr": 2e-4,
                            "peak_vram_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2),
                            "sec_per_step": round(sum(gaps) / len(gaps), 1) if gaps else None,
                            "seconds": round(time.time() - t0), "loss_curve": losses,
                            "nan_loss": any(l != l for l in losses), "lora": "r16 a16 all linear (language only)",
                            "compute_dtype": str(compute_dtype)}
    except Exception as e:  # OOM or a training error is a result here, not a crash
        R["qlora_smoke"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:300]}", "steps": SMOKE_STEPS,
                            "steps_done": len(step_times), "compute_dtype": str(compute_dtype),
                            "peak_vram_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2)}
    save_results(SLUG, R)
print("QLORA", R["qlora_smoke"])

In [ ]:
missing = bl.missing_sections(R, BASE)  # Run all again (after Restart session) fills what is missing
print("BASELINE_DONE" if not missing else f"BASELINE_INCOMPLETE {missing}", SLUG,
      json.dumps({k: R[k] for k in R if k not in ("greetings",)}, ensure_ascii=False))